# 7. Analysis: trade studies, sizing, and requirement consistency

`sysml2.analysis` gives executable models analytical teeth by projecting
them onto external solvers. Three submodules, each behind its own extra
(the core package stays dependency-light):

| Module | Solver | Install | Answers |
|---|---|---|---|
| `analysis.trades` | OR-Tools CP-SAT | `pip install "longeron[trades]"` | which discrete component mixes are feasible / optimal, and why not |
| `analysis.mdao` | OpenMDAO | `pip install "longeron[mdao]"` | continuous sizing, what-ifs, gradient-based optimization |
| `analysis.smt` | Z3 | `pip install "longeron[smt]"` | is the requirement set consistent at all, which requirements conflict, exact feasibility bounds |

The interpreter remains the single source of semantics: every solver
result below is re-evaluated (or cross-checked) against the model itself.

In [ ]:
import sysml2
from sysml2.analysis import mdao, smt, trades

## A component catalog under trade

`examples/drone_catalog.sysml` models a quad-copter as an assembly whose
parts are typed by `variation` definitions: each variation point selects
exactly one `variant` from a catalog (a `[4]` multiplicity selects
homogeneously -- all four motors are the same type). Derived attributes
(mass, cost, thrust, endurance) and `assert constraint` compatibility
rules + system requirements are ordinary SysML.

In [ ]:
catalog = sysml2.load("../examples/drone_catalog.sysml")
study = trades.TradeStudy(catalog, "DroneCatalog::TradeQuad")
for point in study.points.values():
    print(f"{point.name}[{point.count}]: {', '.join(point.variants)}")

In [ ]:
import math

print("candidate mixes:", math.prod(len(p.variants)
                                    for p in study.points.values()))
print("metrics:    ", [name for name, _ in study.derived_order])
print("constraints:", study.constraint_names)

## Enumerate the feasible architectures

`TradeStudy` encodes variant selections as CP-SAT Booleans, variant
attributes as exact fixed-point integers, and the `assert constraint`s as
linear constraints. Every architecture the solver reports is then
re-evaluated *exactly* by the interpreter (`verified`), so fixed-point
rounding can never misreport a design. Of the 54 candidate mixes, only 8
survive the compatibility rules and requirements:

In [ ]:
archs = study.enumerate()
print(f"{len(archs)} of 54 mixes are feasible\n")
for a in sorted(archs, key=lambda a: a.metrics["totalCost"]):
    print(a)

## Optimize a metric

`minimize`/`maximize` run CP-SAT's optimizer over one derived attribute
(returning `None` if nothing is feasible):

In [ ]:
print("cheapest:     ", study.minimize("totalCost"))
print("longest hover:", study.maximize("hoverMinutes"))

## The Pareto front

With several competing objectives, `trades.pareto` filters an enumeration
down to the non-dominated set under *those* objectives. Over (min cost,
min mass, max hover) two mixes survive: the $118 cruiser and the 0.9 kg
racer. Note what earns the racer its place -- it is $41 dearer and hovers
8 minutes *less* than the cruiser; it stays on this front through mass
alone:


In [ ]:
front = trades.pareto(archs, minimize=("totalCost", "totalMass"),
                      maximize=("hoverMinutes",))
for a in front:
    print(a)

### Seeing the whole trade space

`analysis.viz` (the `viz` extra: matplotlib + anywidget) turns those
printouts into pictures. `all_architectures()` evaluates every candidate
mix -- feasible or not -- through the interpreter, and `viz.pareto_figure`
computes the frontier *for the two axes it plots* (here min cost, max
hover) from the feasible mixes. It deliberately does not accept a
precomputed front: projecting the 3-objective front above onto these two
axes would drape a "frontier" over the racer, which is strictly worse on
both plotted metrics. What to look for: the cost-endurance frontier is a
single accent point -- the $118 cruiser is at once the cheapest feasible
mix and the longest-hovering -- and the mass panel below shows the
racer's only claim to fame.


In [ ]:
from sysml2.analysis import viz

mixes = study.all_architectures()          # all 54, interpreter-exact
cruiser = min(front, key=lambda a: a.metrics["totalCost"])
racer = min(front, key=lambda a: a.metrics["totalMass"])
fig = viz.pareto_figure(
    mixes, x="totalCost", y="hoverMinutes", panel_y="totalMass",
    xlabel="total cost (USD)", ylabel="hover endurance (min)",
    panel_ylabel="total mass (kg)",
    annotate={"$118 cruiser: cheapest and longest hover": cruiser,
              "0.9 kg racer: wins only on mass": racer},
    title="One mix wins the cost-endurance trade outright")

### Brushing the mix table

`viz.parcoords` is a small anywidget (built like the replay widget:
inline vanilla JS, no front-end build) -- one line per mix across
categorical axes (the four selections) and numeric axes (the metrics).
Brush gestures live in a narrow zone around each axis (the cursor turns
to a crosshair there); lines stay hoverable everywhere else. Drag along
an axis to brush a range: lines outside any brush fade, and the
surviving row indices sync back to Python through the `selected`
traitlet (`pc.selected_indices()`). A brush is editable after creation:
drag its body to slide the whole interval, drag an end handle to extend
or contract it, click the axis outside the brush (or double-click) to
clear it. Infeasible mixes are the dashed gray lines -- brush
`thrustToWeight` high to watch them drop out.


In [ ]:
rows = viz.mix_table(study, mixes, derived={
    "thrustToWeight": lambda a: a.metrics["totalThrust"]
                                / (a.metrics["totalMass"] * 9.81)})
pc = viz.parcoords(rows, axes=["motors", "props", "battery", "esc",
                               "totalCost", "totalMass",
                               "thrustToWeight", "hoverMinutes"])
pc

## When nothing works: explaining infeasibility

`PicnicQuad :> TradeQuad` adds two more requirements: `longEndurance`
(`hoverMinutes >= 25`) and `cheap` (`totalCost <= 130`). No catalog mix
satisfies them, and `explain()` asks CP-SAT which constraint subset is
already sufficient for infeasibility. The best endurance any mix achieves
is 15 minutes, so `longEndurance` is impossible on its own -- the cost cap
is not even needed:

In [ ]:
picnic = trades.TradeStudy(catalog, "DroneCatalog::PicnicQuad")
print("feasible mixes:", len(picnic.enumerate()))
print("conflict:      ", picnic.explain())

## Continuous sizing with OpenMDAO

Trades pick *which* components; `analysis.mdao` sizes the continuous
attributes of a chosen design. `build_problem` mirrors a part tree onto an
OpenMDAO `Problem`: nested parts become `Group`s, literal-valued
attributes become `IndepVarComp` outputs (candidate design variables),
derived attributes become components that evaluate through the
interpreter, and each constraint or requirement becomes a `*_margin`
output that is `>= 0` iff it holds.

In [ ]:
drone = sysml2.load("../examples/drone.sysml")
build = mdao.build_problem(drone, "Drone::QuadCopter",
                           requirements=("Drone::FlightEnvelope",))
p = build.problem
p.run_model()
print("totalMass:              ", p.get_val("totalMass")[0])
print("takeoffMassLimit margin:", p.get_val("takeoffMassLimit_margin")[0])
print("canHover margin:        ", p.get_val("canHover_margin")[0])
print("hoverMargin margin:     ", p.get_val("hoverMargin_margin")[0])

The `hoverMargin` requirement evaluates *through* the `ThrustToWeight`
calc definition. What-ifs are one `set_val` away -- a 0.9 kg payload blows
the 1.5 kg takeoff limit:

In [ ]:
p.set_val("payloadMass", 0.9)
p.run_model()
print("totalMass:              ", p.get_val("totalMass")[0])
print("takeoffMassLimit margin:", p.get_val("takeoffMassLimit_margin")[0])

## Optimization: the largest payload that still flies

`add_optimization` wires up SLSQP with every margin as a `>= 0`
constraint. Maximizing `payloadMass` drives the takeoff-mass limit to its
bound: 0.46 kg of payload at exactly 1.5 kg total.

In [ ]:
opt = mdao.build_problem(drone, "Drone::QuadCopter", setup=False,
                         requirements=("Drone::FlightEnvelope",))
mdao.add_optimization(opt, objective="payloadMass",
                      design_vars={"payloadMass": (0.0, 3.0)},
                      maximize=True)
opt.problem.setup()
opt.problem.set_val("payloadMass", 0.1)
opt.problem.run_driver()
print(f"max payload = {opt.problem.get_val('payloadMass')[0]:.2f} kg "
      f"(totalMass = {opt.problem.get_val('totalMass')[0]:.2f} kg)")

### The margin picture

Sweeping `payloadMass` through the built problem shows *why* 0.46 kg is
the ceiling. `viz.margin_sweep_figure` re-runs the model across the
sweep and plots every requirement margin (each is >= 0 iff its
constraint holds). What to look for: the shaded feasible band ends
exactly where the takeoff-mass limit crosses zero -- it is the binding
requirement -- while both hover margins stay comfortable throughout.


In [ ]:
fig = viz.margin_sweep_figure(
    build.problem, "payloadMass", [i / 50 for i in range(51)],
    build.constraints, xlabel="payload mass (kg)",
    title="Payloads above 0.46 kg cannot fly")

## Requirement consistency with Z3

`analysis.smt` works over *unbounded reals*, so it answers questions the
other two cannot: is a requirement set satisfiable at all? `to_smt`
encodes the part tree's attributes as Z3 constants, value expressions as
equalities, and constraint/requirement bodies as labeled assertions
(calc invocations are inlined). `check()` returns a witness when
consistent:

In [ ]:
system = smt.to_smt(drone, "Drone::QuadCopter",
                    requirements=("Drone::FlightEnvelope",))
result = system.check()
print(result.status)
print({k: result.witness[k] for k in ("payloadMass", "totalMass")})

Listing an attribute in `free=` drops its defining equation, turning it
into a degree of freedom -- and `maximize` reports *exact* rational
bounds. The feasible payload ceiling is 23/50 = 0.46 kg: precisely the
value SLSQP converged to above, and exactly where the margin sweep's
feasible band ends -- one chart, one number, one story:


In [ ]:
freed = smt.to_smt(drone, "Drone::QuadCopter",
                   requirements=("Drone::FlightEnvelope",),
                   free=("payloadMass",))
bound, _ = freed.maximize("payloadMass")
print("max feasible payloadMass =", bound, "kg")

## Conflicting requirements: the unsat core

Now add a requirement that cannot coexist with the takeoff-mass limit --
a 0.6 kg payload demand -- using programmatic authoring (notebook 1):

In [ ]:
from sysml2 import model as M

heavy = M.Definition(kind="requirement", name="HeavyPayload")
heavy.add(M.Usage(kind="subject", name="drone", types=["QuadCopter"]))
heavy.add(M.Usage(
    kind="constraint", name="bigPayload", constraint_kind="require",
    result=sysml2.parse_expression("drone.payloadMass >= 0.6")))
drone.find("Drone").add(heavy)

In [ ]:
conflicted = smt.to_smt(drone, "Drone::QuadCopter",
                        requirements=("Drone::FlightEnvelope",
                                      "Drone::HeavyPayload"),
                        free=("payloadMass",))
result = conflicted.check()
print(result.status)
for label in result.core:
    print(" ", label)

The core names exactly the conflicting subset: `bigPayload` collides with
`takeoffMassLimit` through the two defining equations -- and correctly
*excludes* the (satisfiable) `canHover` constraint, so you know which
requirement to renegotiate.

## Where this goes: the two-level loop

The three solvers compose into classic mixed-discrete MDO: **CP-SAT picks
the architecture, OpenMDAO sizes it** -- `enumerate()`/`pareto()` yield
component mixes, each mix binds the variation points to concrete part
definitions, `build_problem` optimizes the continuous attributes, and the
exact interpreter metrics feed back into the trade ranking -- while Z3
guards the requirement set for consistency before you spend solver time
on an impossible problem.

## From mixes to shapes: 3D design views

Numbers pick the mix; geometry shows what you picked.
`analysis.geometry` builds a to-scale drone from a mix's catalog
attributes -- the frame is *derived* from prop diameter + tip clearance,
motor cylinders from motor mass, the battery box from battery mass --
and `analysis.viewer3d` renders the baked meshes with three.js in a
small anywidget. House pattern throughout: Python bakes the geometry
once per mix (< 1 ms, no CAD kernel); the front-end only paints. Drag
to orbit, scroll to zoom, double-click to re-fit. (The front-end loads
three.js from a CDN at view time, so this one view needs network
access.)

The cruiser again -- upgraded to the 10-inch props its motors can
swing, for four more dollars -- next to the 0.9 kg racer, at true
scale:

In [ ]:
from sysml2.analysis import geometry, viewer3d

cruiser10 = study.evaluate({"motors": "sunnySky2212", "props": "apc1045",
                            "battery": "lipo3s2200", "esc": "esc20"})
viewer3d.mesh_viewer(
    geometry.architecture_geometry(study, cruiser10),
    geometry.architecture_geometry(study, racer),
    label=f"10-in cruiser: ${cruiser10.metrics['totalCost']:g}, "
          f"{cruiser10.metrics['hoverMinutes']:g} min",
    label_b=f"5-in racer: ${racer.metrics['totalCost']:g}, "
            f"{racer.metrics['totalMass']:g} kg")

### Linked selection: parallel coordinates -> 3D

The widgets compose through plain traitlets: observe `selected` on the
parallel-coordinates widget and re-bake the first surviving mix into the
viewer. Brush the cost axis above down to its cheap end and this drone
re-sizes to match.

In [ ]:
import json

linked = viewer3d.mesh_viewer(
    geometry.architecture_geometry(study, cruiser10),
    label=" / ".join(cruiser10.selection.values()))

def show_first_selected(change):
    indices = json.loads(change["new"] or "[]")
    if indices:
        mix = mixes[indices[0]]
        linked.mesh_json = json.dumps(
            geometry.architecture_geometry(study, mix))
        linked.label = " / ".join(mix.selection.values())

pc.observe(show_first_selected, names="selected")
linked

### Real CAD when you need it

For actual CAD output -- STEP for a printable frame -- `geometry.
to_cadquery` rebuilds the same parametric assembly as cadquery solids,
behind the `cad` extra (the OCC kernel is ~1 GB, which is why the mesh
pipeline above never touches it). This cell degrades to a note when
cadquery is not installed:

In [ ]:
try:
    assembly = geometry.to_cadquery(
        **geometry.architecture_params(study, cruiser10))
    print(f"cadquery assembly with {len(assembly.children)} parts -- "
          "assembly.export('drone.step') exports STEP")
except ImportError:
    print('optional: pip install "longeron[cad]" enables STEP export '
          "-- skipping here")